# Getting started with MemsArray object

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

from megamicros.log import log
from megamicros.core.base import MemsArray
from megamicros.bmf import Beamformer

SAMPLING_FREQUENCY = 10000

log.setLevel( "INFO" )

# Declare a 32 MEMs antenna
antenna = MemsArray( available_mems_number=32 )

# verify available mems number
print( f"available mems number={antenna.available_mems_number}" )

# set active mems
antenna.setActiveMems( [i for i in range(32)] )
print( f"active mems number={antenna.mems_number}" )

# iterate over the antenna data stream
for i, data in enumerate( antenna ):
    print( f"data={data}")
    if i > 10:
        break

mems = np.load ('Antenna-square-JetsonNano-0001.npy' )

bmf = Beamformer( 
    mems_position = mems,
    sampling_frequency = SAMPLING_FREQUENCY,
    window_size = 256,    
    area = [5, 5, 0.01],
    area_quantization = [2, 2.1, 1/0.01]
)

bmf.setBandWidth( [100, 800], unit="frequency" )

bmf.moveArea( [0, 0, -2] )
bmf.init()

print( f"Mems number is : {bmf.getMemsNumber()}" )

# print area locations and antenna 
space_locations = bmf.getLocations()
mems_location = bmf.getMems()

fig = plt.figure()
ax = fig.add_subplot( 121, projection='3d' )
ax.scatter( space_locations[:,0], space_locations[:,1], space_locations[:,2] )
ax.scatter( mems_location[:,0], mems_location[:,1], mems_location[:,2], marker='^' )
ax = fig.add_subplot( 122, projection='3d' )
ax.scatter( mems_location[:,0], mems_location[:,1], mems_location[:,2], marker='^' )
fig.show()

signal = np.zeros( (256, 25) )
bf = bmf.beamform( signal )


## Getting signal from DB

In [ ]:
from megamicros.db.query import AidbSession
from megamicros.data import MuAudio

# Available labels
LABEL_SOW_FEEDING_CALL = 18
LABEL_PIGLET_SQUEALS = 15
LABEL_SOW_GRUNT_NERVOUS = 16
LABEL_ROOM_NOISE = 29
LABEL_SOW_GRUNT = 8
LABEL_SOW_GRUNT_MODSTRESS  = 1
LABEL_SOW_SCREAMS = 3
LABEL_PIGLET_SQUEALS_2 = 5

# choose your label:
LABEL_ID = LABEL_SOW_GRUNT_NERVOUS

with AidbSession(
    dbhost='http://dbwelfare.biimea.io/',
    login='ailab',
    email='bruno.gas@biimea.com',
    password='#T;uZnQ5UJ_JC~&' ) as session:
        label_id = LABEL_ID
        labelings_file = session.load_labelings( label_id=label_id )
        print( f"Fichiers trouvés:" )
        for labeling_file in labelings_file:
            print( f" > [{labeling_file['sourcefile_id']}: {labeling_file['sourcefile_filename']}]")
        sourcefile_id = int( input( 'Numéro identificateur du fichier à sélectionner:' ) )
        limit = 100
        channels = list( np.arange( 32 ) + 1 )
        signals = session.load_labelized( sourcefile_id=sourcefile_id, label_id=label_id, limit=limit, channels=channels )

In [ ]:
# Choose your file section
print( f"{len(signals)} section audio récupérées: " )
for idx, aud in enumerate( signals ):
    print( f"Audio[{idx}]: {aud} -> label={aud.label}, channels number: {aud.channels_number} ({aud.samples_number} samples)")
selected = int( input( f"Choisir un signal (0 à {len(signals)-1}): " ) )
signal: MuAudio = signals[selected]

In [ ]:
# get infos
print( f"label={signal.label}" )
print( f"channels_number={signal.channels_number}" )
print( f"samples_number={signal.samples_number}" )
print( f"sampling_frequency={signal.sampling_frequency}" )

# Play sound using channel 0 and 1
left = np.array( signal.channel(0) )
right = np.array( signal.channel(1) )
sound = np.array( [left, right] )

In [ ]:
from IPython import display
display.Audio(sound, rate=SAMPLING_FREQUENCY )

In [ ]:
# Get the whole 32 channels signal as a numpy.ndarray
my_signal = signal()
print( np.shape(my_signal) )